# Reddit thread collection and preparation 

**Author(s):** Aayah Osman 

**Source:** Reddit DuckDB database from class that we got on the Wellesley HPC cluster. The database is not included in the repository because it is 2 GB.


## 1. Set up paths and identify thread IDs

For this part, I started with nine Reddit threads that I had found about beauty standards in K-dramas, especially discussions related to True Beauty and My ID Is Gangnam Beauty. I saved the thread IDs in a JSONL file so I could use them to find the corresponding posts and comments in the Reddit database provided in class. I also set up the paths to the database and extraction script on the Wellesley HPC cluster.

In [ ]:
from pathlib import Path
import json
import pandas as pd

folder = Path.cwd()
db_path = Path("/home/shared/cs315/reddit-data/kdrama_reddit.duckdb")
script_folder = Path("/home/shared/cs315/reddit-data")

thread_ids = [
    "1trhwfe", "1r942db", "1fw4m2p", "1gh646v", "r5fv5n",
    "qtuyvx", "xa2t1v", "1tijc0s", "gsepuk"
]

selection_file = folder / "selected_threads.jsonl"
with selection_file.open("w", encoding="utf-8") as f:
    for thread_id in thread_ids:
        f.write(json.dumps({"thread_identifier": f"t3_{thread_id}"}) + "\n")
print(f"Saved {len(thread_ids)} requested IDs to {selection_file}")

## 2. Extract matching threads from the course database

I used the extract_thread_bodies.py script provided in class to look for the nine selected threads in the Reddit DuckDB database. The script searches for each thread ID and extracts the original post and its comments into a separate JSONL file.

Out of the nine threads we selected, only four were found in the database. The five missing threads were all from the r/truebeautydrama subreddit. This is because the database does not include that subreddit. I continued working with the four threads that were available.

In [ ]:
import sys

if not db_path.exists():
    print("Course database not found here. Run this cell on the class HPC server.")
else:
    sys.path.insert(0, str(script_folder))
    from extract_thread_bodies import export_threads_to_jsonl
    export_threads_to_jsonl(
        input_file=str(selection_file),
        db_path=str(db_path),
        output_dir=str(folder / "exported_threads"),
    )

exported_files = sorted((folder / "exported_threads").glob("*.jsonl"))
found_ids = {f.stem.removeprefix("t3_") for f in exported_files}
print("Retrieved thread IDs:", sorted(found_ids))
print("Missing thread IDs:", sorted(set(thread_ids) - found_ids))

## 3. Convert retrieved JSONL records to CSV

The extraction script saved the retrieved threads as JSONL files, so I used pandas to convert each file into a CSV. I also combined the four threads into one dataset to make it easier to work with all the comments together.

The combined dataset first had 396 records. This included both the original Reddit posts and their comments. I kept these records together at first because I still needed information from the original posts, especially the thread titles, before creating a separate dataset containing only comments.

In [ ]:
csv_folder = folder / "reddit_csv"
csv_folder.mkdir(exist_ok=True)
frames = []
for file in exported_files:
    part = pd.read_json(file, lines=True)
    part["thread_id"] = file.stem.removeprefix("t3_")
    part.to_csv(csv_folder / f"{file.stem}.csv", index=False)
    frames.append(part)

if not frames:
    raise RuntimeError("No extracted threads to process.")

raw = pd.concat(frames, ignore_index=True)

raw.to_csv(folder / "reddit_all_threads.csv", index=False)
print("All records:", len(raw))
print("Fields:", raw.columns.tolist())
print("Records per thread:")
print(raw["thread_id"].value_counts())

## 4. Create an analysis-ready, comment-only CSV

Then I cleaned the dataset so that each row represented an individual Reddit comment rather than an original post. I removed comments that were missing text or had been deleted or removed. I also added the original thread title to each comment and labeled the threads based on whether they were about True Beauty, My ID Is Gangnam Beauty, or beauty standards in K-dramas more generally.

I converted the Reddit timestamps into readable dates and extracted the year each comment was posted. I also removed the author information. Finally, I checked for duplicate comment IDs and saved the cleaned dataset as a new CSV.

After cleaning, I had 368 comments with no missing values in the columns I kept.

In [ ]:
titles = (
    raw.dropna(subset=["title"])
       .drop_duplicates("thread_id")
       .set_index("thread_id")["title"]
       .to_dict()
)
comments = raw[raw["comment_id"].notna()].copy()
comments = comments.dropna(subset=["comment_body"])
comments = comments[~comments["comment_body"].str.strip().isin(["", "[deleted]", "[removed]"])].copy()
comments["thread_title"] = comments["thread_id"].map(titles)

# The general thread is not specific to any single kdrama.
dramas = {
    "1trhwfe": "General K-drama discussion",
    "1tijc0s": "True Beauty",
    "xa2t1v": "My ID Is Gangnam Beauty",
    "gsepuk": "My ID Is Gangnam Beauty",
}
comments["drama"] = comments["thread_id"].map(dramas).fillna("Unclassified")


comments["date_posted"] = pd.to_datetime(comments["created_utc"], unit="s", utc=True, errors="coerce")
comments["year_posted"] = comments["date_posted"].dt.year
comments["text"] = comments["comment_body"]

# Need to remove the author field and profile links from our final data.
clean = comments[[
    "comment_id", "thread_id", "thread_title", "subreddit", "drama",
    "date_posted", "year_posted", "text", "score"
]].drop_duplicates(subset=["comment_id"])
clean.to_csv(folder / "reddit_comments_clean.csv", index=False)
print("Clean comment records:", len(clean))
print("Missing values:\n", clean.isna().sum())

## 5. Preliminary data checks

I looked at how many comments came from each thread and drama, when the comments were posted, and how long they were on average and mst of the comments came from the general beauty standards discussion, which had 306 comments. The two threads about My ID Is Gangnam Beauty had 47 comments combined, while the True Beauty thread had 15. The comments were posted between 2020 and 2026, ith 321 comments from 2026, 32 from 2022, and 15 from 2020. The average comment was about 39 words long, and there were no duplicate comment IDs. It was interesting how unevenly the comments were distributed across the threads. Since most of the data comes from one general discussion, we are going to need to be careful about how we compare audience discussions across different dramas.

In [ ]:
print("Comments by thread:\n", clean["thread_id"].value_counts())
print("Comments by drama:\n", clean["drama"].value_counts())
print("Comments by year:\n", clean["year_posted"].value_counts().sort_index())
print("Duplicate IDs:", clean["comment_id"].duplicated().sum())
print("Earliest:", clean["date_posted"].min())
print("Latest:", clean["date_posted"].max())
print("Average comment length (words):", round(clean["text"].str.split().str.len().mean(), 1))

## 6. Attempt to Get Some Visualizations

I created two bar charts to get a better idea of how the Reddit comments were distributed. The first shows the number of comments associated with each drama or general discussion, and the second shows the number of comments posted each year. The first chart makes it clear that most of the comments came from the general K-drama discussion rather than the threads about individual shows. The second shows that most of the comments were posted in 2026. These visualizations are helpful for understanding the dataset before moving into the actual text analysis. They also show why we need to be careful about comparing the dramas, especially because we don't have the same amount of data for each one.



In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

figure_dir = folder.parent / "visualizations"
figure_dir.mkdir(exist_ok=True)

for field, title, filename in [
    ("drama", "Reddit comments by drama / discussion", "reddit_comments_by_drama.png"),
    ("year_posted", "Reddit comments by posting year", "reddit_comments_by_year.png"),
]:
    clean[field].value_counts().sort_index().plot(kind="bar")
    plt.title(title)
    plt.xlabel(field.replace("_", " ").title())
    plt.ylabel("Number of comments")
    plt.xticks(rotation=30 if field == "drama" else 0, ha="right" if field == "drama" else "center")
    plt.tight_layout()
    plt.savefig(figure_dir / filename, dpi=200)
    plt.close()
    print("Saved", figure_dir / filename)

## 7. Interpretation, limitations, and next steps

Overall, I was able to collect and clean 368 Reddit comments from four threads that we had selected for our project. The comments include discussions about beauty standards, makeup, plastic surgery, insecurities, and also just generally how K-dramas communicate messages about beauty. One of the biggest limitations is that I could only get four of the nine threads we originally had. But that is an easy fix. However this does mean that some discussions I wanted to include from r/truebeautydrama are missing. Another limitation is that most of the comments come from one general discussion about K-dramas, so the dataset isn't evenly distributed across the different shows. We are also going to need to be careful when it comes to distinguishing between the year a drama was released and the year a comment was posted. Like a comment posted in 2026 about an older drama doesn't necessarily tell us what audiences thought when that drama originally came out.

Moving forward though, I want to start examining which comments actually discuss beauty standards and what specific themes come up most frequently. We plan to use an inductive approach so that our categories are based on what audiences are actually saying rather than deciding all the categories beforehand. After developing the initial themes, we can use GenAI to help label a larger number of comments.
